# 03 — Reward Model Training

**RLHF Pipeline for Compact Open-Source LLMs**

This notebook trains a scalar reward model from human preference pairs. The reward model learns to predict which response a human would prefer, and its scores are used to guide PPO-based RLHF training.

## Design Choices

- **Architecture:** Same base model (Qwen2.5-0.5B) with a classification head (num_labels=1)
- **Fine-tuning:** LoRA with SEQ_CLS task type
- **Training data:** Tokenized preference pairs from HH-RLHF
- **Framework:** TRL RewardTrainer
- **Key metric:** Pairwise preference accuracy (does the model assign higher score to chosen?)

## 3.1 Environment and Imports

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", palette="muted")

from src.data_utils import set_seed, load_dataset_from_disk
from src.model_utils import load_reward_model, load_tokenizer, apply_lora, build_lora_config
from src.reward_train import prepare_reward_dataset, compute_reward_scores
from src.metrics import pairwise_preference_accuracy, compute_reward_model_metrics, reward_score_distribution

set_seed(42)
print(f"CUDA: {torch.cuda.is_available()}")

In [ ]:
# Configuration
MODEL_NAME     = "Qwen/Qwen2.5-0.5B"
MAX_SEQ_LENGTH = 512
OUTPUT_DIR     = PROJECT_ROOT / "outputs" / "models" / "reward_model_hh"
LOG_DIR        = PROJECT_ROOT / "outputs" / "logs"
FIGURES_DIR    = PROJECT_ROOT / "outputs" / "figures"
TABLES_DIR     = PROJECT_ROOT / "outputs" / "tables"

for d in [OUTPUT_DIR, LOG_DIR, FIGURES_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

cfg = {
    "seed": 42,
    "training": {"fp16": True, "bf16": False, "gradient_checkpointing": True},
    "lora": {
        "r": 16, "lora_alpha": 32, "lora_dropout": 0.05,
        "target_modules": ["q_proj", "v_proj"], "bias": "none",
    },
}

## 3.2 Load Preference Data

In [ ]:
pref_train = load_dataset_from_disk(PROJECT_ROOT / "data" / "processed" / "hh_rlhf" / "train")
pref_val   = load_dataset_from_disk(PROJECT_ROOT / "data" / "processed" / "hh_rlhf" / "validation")

print(f"Train: {len(pref_train)} pairs, columns: {pref_train.column_names}")
print(f"Val:   {len(pref_val)} pairs")

## 3.3 Tokenize for Reward Trainer

TRL RewardTrainer expects columns: `input_ids_chosen`, `attention_mask_chosen`, `input_ids_rejected`, `attention_mask_rejected`.

In [ ]:
tokenizer = load_tokenizer(MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH)

train_tok = prepare_reward_dataset(pref_train, tokenizer, MAX_SEQ_LENGTH)
val_tok   = prepare_reward_dataset(pref_val,   tokenizer, MAX_SEQ_LENGTH)

print(f"Tokenized train columns: {train_tok.column_names}")
print(f"Tokenized val columns:   {val_tok.column_names}")
print(f"Chosen token length:     {len(train_tok[0]['input_ids_chosen'])}")

## 3.4 Model Setup

In [ ]:
model = load_reward_model(MODEL_NAME, cfg, num_labels=1)

lora_config = build_lora_config(cfg, task_type="SEQ_CLS")
model = apply_lora(model, lora_config)

# PEFT + gradient checkpointing compatibility
use_grad_ckpt = torch.cuda.is_available()
if use_grad_ckpt:
    model.enable_input_require_grads()
model.train()

## 3.5 Reward Model Training

In [ ]:
from transformers import TrainingArguments
from trl import RewardTrainer

use_fp16 = cfg["training"]["fp16"] and torch.cuda.is_available()

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    logging_steps=50,
    save_steps=500,
    eval_steps=250,
    evaluation_strategy="steps",
    save_total_limit=2,
    fp16=use_fp16,
    gradient_checkpointing=use_grad_ckpt,
    report_to="none",
    seed=42,
    remove_unused_columns=False,  # RewardTrainer needs _chosen/_rejected columns
)

trainer = RewardTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
)

print("RewardTrainer created. Starting training...")

In [ ]:
# Run this cell to start actual training
trainer.train()

# Save model and log
model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

pd.DataFrame(trainer.state.log_history).to_csv(LOG_DIR / "reward_training_log.csv", index=False)
print(f"Reward model saved to {OUTPUT_DIR}")

## 3.6 Training Loss Curve

In [ ]:
log_path = LOG_DIR / "reward_training_log.csv"

if log_path.exists():
    log_df = pd.read_csv(log_path)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    train_rows = log_df.dropna(subset=["loss"])
    if not train_rows.empty:
        axes[0].plot(train_rows["step"], train_rows["loss"])
        axes[0].set_xlabel("Step")
        axes[0].set_ylabel("Loss")
        axes[0].set_title("Reward Model Training Loss")
    
    eval_rows = log_df.dropna(subset=["eval_loss"])
    if not eval_rows.empty:
        axes[1].plot(eval_rows["step"], eval_rows["eval_loss"], marker="o", markersize=4, color="orange")
        axes[1].set_xlabel("Step")
        axes[1].set_ylabel("Eval Loss")
        axes[1].set_title("Reward Model Eval Loss")
    
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "reward_loss_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("TO BE FILLED AFTER RUNNING REWARD MODEL TRAINING")

## 3.7 Validation — Pairwise Preference Accuracy

In [ ]:
if OUTPUT_DIR.exists() and (OUTPUT_DIR / "adapter_config.json").exists():
    model.eval()
    
    chosen_texts   = [f"{p}\n\n{c}" for p, c in zip(pref_val["prompt"], pref_val["chosen"])]
    rejected_texts = [f"{p}\n\n{r}" for p, r in zip(pref_val["prompt"], pref_val["rejected"])]
    
    chosen_scores   = compute_reward_scores(model, tokenizer, chosen_texts,   MAX_SEQ_LENGTH, batch_size=8)
    rejected_scores = compute_reward_scores(model, tokenizer, rejected_texts, MAX_SEQ_LENGTH, batch_size=8)
    
    metrics = compute_reward_model_metrics(chosen_scores, rejected_scores)
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
    
    pd.DataFrame([metrics]).to_csv(TABLES_DIR / "reward_model_val_metrics.csv", index=False)
else:
    print("TO BE FILLED AFTER RUNNING REWARD MODEL TRAINING")

## 3.8 Chosen vs Rejected Score Distribution

In [ ]:
if OUTPUT_DIR.exists() and (OUTPUT_DIR / "adapter_config.json").exists():
    score_df = reward_score_distribution(chosen_scores, rejected_scores)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].hist(score_df["chosen_score"], bins=30, alpha=0.7, label="Chosen", color="steelblue")
    axes[0].hist(score_df["rejected_score"], bins=30, alpha=0.7, label="Rejected", color="salmon")
    axes[0].set_xlabel("Reward Score")
    axes[0].set_ylabel("Count")
    axes[0].set_title("Chosen vs Rejected Score Distribution")
    axes[0].legend()
    
    axes[1].hist(score_df["gap"], bins=30, color="mediumpurple", alpha=0.8)
    axes[1].axvline(x=0, color="red", linestyle="--", alpha=0.5)
    axes[1].set_xlabel("Score Gap (Chosen - Rejected)")
    axes[1].set_ylabel("Count")
    axes[1].set_title("Reward Score Gap Distribution")
    
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "reward_score_distribution.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("TO BE FILLED AFTER RUNNING REWARD MODEL TRAINING")

## 3.9 Reward Inference Examples

In [ ]:
if OUTPUT_DIR.exists() and (OUTPUT_DIR / "adapter_config.json").exists():
    sample_texts = [
        "Human: What is the capital of France?\n\nAssistant: The capital of France is Paris.",
        "Human: What is the capital of France?\n\nAssistant: I don't know.",
        "Human: Tell me a joke.\n\nAssistant: Why did the chicken cross the road? To get to the other side!",
        "Human: How do I cook pasta?\n\nAssistant: Boil water, add salt, cook pasta for 8-10 minutes, drain.",
    ]
    scores = compute_reward_scores(model, tokenizer, sample_texts, MAX_SEQ_LENGTH, batch_size=4)
    
    print("Reward inference examples:")
    for text, score in zip(sample_texts, scores):
        assistant_part = text.split("Assistant: ", 1)[-1][:80]
        print(f"  score={score:.4f}  |  {assistant_part}")
else:
    print("TO BE FILLED AFTER RUNNING REWARD MODEL TRAINING")

## Summary

| Output | Location |
|---|---|
| Reward model LoRA adapter | `outputs/models/reward_model_hh/` |
| Training log CSV | `outputs/logs/reward_training_log.csv` |
| Loss curve plot | `outputs/figures/reward_loss_curves.png` |
| Score distribution plot | `outputs/figures/reward_score_distribution.png` |
| Validation metrics CSV | `outputs/tables/reward_model_val_metrics.csv` |

**Next:** Proceed to `04_rlhf_ppo.ipynb` for PPO-based RLHF training.